In [1]:
from pyspark import SparkContext

In [2]:
sc = SparkContext.getOrCreate()

In [3]:
movies_rdd = sc.textFile("demo_dir/movies.csv")

['movieId,title,genres', '1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy', '2,Jumanji (1995),Adventure|Children|Fantasy', '3,Grumpier Old Men (1995),Comedy|Romance', '4,Waiting to Exhale (1995),Comedy|Drama|Romance']


In [4]:
movies_rdd.take(5)

['movieId,title,genres',
 '1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy',
 '2,Jumanji (1995),Adventure|Children|Fantasy',
 '3,Grumpier Old Men (1995),Comedy|Romance',
 '4,Waiting to Exhale (1995),Comedy|Drama|Romance']

In [5]:
header = movies_rdd.first()
movies_data = movies_rdd.filter(lambda row: row != header)

In [6]:
movies_data.take(5)

['1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy',
 '2,Jumanji (1995),Adventure|Children|Fantasy',
 '3,Grumpier Old Men (1995),Comedy|Romance',
 '4,Waiting to Exhale (1995),Comedy|Drama|Romance',
 '5,Father of the Bride Part II (1995),Comedy']

In [19]:
import csv

movies_split = movies_data.map(lambda row: next(csv.reader([row])))

In [20]:
movies_split.take(5)

[['1', 'Toy Story (1995)', 'Adventure|Animation|Children|Comedy|Fantasy'],
 ['2', 'Jumanji (1995)', 'Adventure|Children|Fantasy'],
 ['3', 'Grumpier Old Men (1995)', 'Comedy|Romance'],
 ['4', 'Waiting to Exhale (1995)', 'Comedy|Drama|Romance'],
 ['5', 'Father of the Bride Part II (1995)', 'Comedy']]

In [9]:
#movies_split = movies_data.map(lambda row: row.split(",", 2))

In [10]:
#movies_split.take(5)

[['1', 'Toy Story (1995)', 'Adventure|Animation|Children|Comedy|Fantasy'],
 ['2', 'Jumanji (1995)', 'Adventure|Children|Fantasy'],
 ['3', 'Grumpier Old Men (1995)', 'Comedy|Romance'],
 ['4', 'Waiting to Exhale (1995)', 'Comedy|Drama|Romance'],
 ['5', 'Father of the Bride Part II (1995)', 'Comedy']]

In [25]:
genre_pairs = movies_split.flatMap(
    lambda row: [(genre, 1) for genre in row[2].split("|")]
)

In [26]:
genre_pairs.take(10)

[('Adventure', 1),
 ('Animation', 1),
 ('Children', 1),
 ('Comedy', 1),
 ('Fantasy', 1),
 ('Adventure', 1),
 ('Children', 1),
 ('Fantasy', 1),
 ('Comedy', 1),
 ('Romance', 1)]

In [23]:
genre_counts = genre_pairs.reduceByKey(lambda x, y: x + y)

In [24]:
genre_counts.collect()

[('Children', 2182),
 ('Fantasy', 2212),
 ('Romance', 6069),
 ('Drama', 19806),
 ('Action', 5775),
 ('Thriller', 6761),
 ('Horror', 4448),
 ('Sci-Fi', 2847),
 ('IMAX', 197),
 ('Documentary', 4122),
 ('Musical', 1079),
 ('Western', 1028),
 ('Crime', 4247),
 ('Film-Noir', 360),
 ('Comedy', 13002),
 ('Mystery', 2274),
 ('War', 1544),
 ('Adventure', 3369),
 ('Animation', 1942),
 ('(no genres listed)', 2756)]

In [27]:
genre_movie_pairs = movies_split.flatMap(
    lambda row: [(genre, row[1]) for genre in row[2].split("|")]
)

In [28]:
genre_movie_pairs.take(10)

[('Adventure', 'Toy Story (1995)'),
 ('Animation', 'Toy Story (1995)'),
 ('Children', 'Toy Story (1995)'),
 ('Comedy', 'Toy Story (1995)'),
 ('Fantasy', 'Toy Story (1995)'),
 ('Adventure', 'Jumanji (1995)'),
 ('Children', 'Jumanji (1995)'),
 ('Fantasy', 'Jumanji (1995)'),
 ('Comedy', 'Grumpier Old Men (1995)'),
 ('Romance', 'Grumpier Old Men (1995)')]

In [30]:
genre_movie_pairs.lookup("Comedy")[:10]

['Toy Story (1995)',
 'Grumpier Old Men (1995)',
 'Waiting to Exhale (1995)',
 'Father of the Bride Part II (1995)',
 'Sabrina (1995)',
 'American President, The (1995)',
 'Dracula: Dead and Loving It (1995)',
 'Four Rooms (1995)',
 'Ace Ventura: When Nature Calls (1995)',
 'Money Train (1995)']

In [32]:
grouped_genres = genre_movie_pairs.groupByKey().mapValues(list)

In [33]:
grouped_genres.take(5)

[('Children',
  ['Toy Story (1995)',
   'Jumanji (1995)',
   'Tom and Huck (1995)',
   'Balto (1995)',
   'Now and Then (1995)',
   'Babe (1995)',
   'It Takes Two (1995)',
   'Pocahontas (1995)',
   'Big Green, The (1995)',
   'Kids of the Round Table (1995)',
   'Indian in the Cupboard, The (1995)',
   'White Balloon, The (Badkonake sefid) (1995)',
   'Dunston Checks In (1996)',
   'Muppet Treasure Island (1996)',
   'NeverEnding Story III, The (1994)',
   'Amazing Panda Adventure, The (1995)',
   'Casper (1995)',
   'Free Willy 2: The Adventure Home (1995)',
   'Mighty Morphin Power Rangers: The Movie (1995)',
   'Far From Home: The Adventures of Yellow Dog (1995)',
   'Goofy Movie, A (1995)',
   'Fluke (1995)',
   'Gordy (1995)',
   'Gumby: The Movie (1995)',
   'Heavyweights (Heavy Weights) (1995)',
   "Kid in King Arthur's Court, A (1995)",
   'Little Princess, A (1995)',
   'Swan Princess, The (1994)',
   'Secret of Roan Inish, The (1994)',
   'Baby-Sitters Club, The (1995)',
  

In [34]:
genre_key_counts = genre_movie_pairs.countByKey()

In [35]:
genre_key_counts

defaultdict(int,
            {'Adventure': 3369,
             'Animation': 1942,
             'Children': 2182,
             'Comedy': 13002,
             'Fantasy': 2212,
             'Romance': 6069,
             'Drama': 19806,
             'Action': 5775,
             'Crime': 4247,
             'Thriller': 6761,
             'Horror': 4448,
             'Mystery': 2274,
             'Sci-Fi': 2847,
             'IMAX': 197,
             'Documentary': 4122,
             'War': 1544,
             'Musical': 1079,
             'Western': 1028,
             'Film-Noir': 360,
             '(no genres listed)': 2756})

In [36]:
genre_pairs.reduceByKey(lambda x, y: x + y)

PythonRDD[36] at RDD at PythonRDD.scala:53

In [38]:
genre_counts.collect()

[('Children', 2182),
 ('Fantasy', 2212),
 ('Romance', 6069),
 ('Drama', 19806),
 ('Action', 5775),
 ('Thriller', 6761),
 ('Horror', 4448),
 ('Sci-Fi', 2847),
 ('IMAX', 197),
 ('Documentary', 4122),
 ('Musical', 1079),
 ('Western', 1028),
 ('Crime', 4247),
 ('Film-Noir', 360),
 ('Comedy', 13002),
 ('Mystery', 2274),
 ('War', 1544),
 ('Adventure', 3369),
 ('Animation', 1942),
 ('(no genres listed)', 2756)]

In [37]:
genre_movie_pairs.countByKey()

defaultdict(int,
            {'Adventure': 3369,
             'Animation': 1942,
             'Children': 2182,
             'Comedy': 13002,
             'Fantasy': 2212,
             'Romance': 6069,
             'Drama': 19806,
             'Action': 5775,
             'Crime': 4247,
             'Thriller': 6761,
             'Horror': 4448,
             'Mystery': 2274,
             'Sci-Fi': 2847,
             'IMAX': 197,
             'Documentary': 4122,
             'War': 1544,
             'Musical': 1079,
             'Western': 1028,
             'Film-Noir': 360,
             '(no genres listed)': 2756})

## Key Insights
- Drama is the most common genre
- Comedy has high frequency
- Some genres like IMAX are rare